# Tracelet crash detection — validation against VZCrash field data

Supports [Tracelet issue #173](https://github.com/Ikolvi/Tracelet/issues/173): validate and tune the
crash/fall detector against **real labelled vehicle data** before promoting it from beta to stable.

Dataset: [`vzc-research-chapter/VZCrash`](https://huggingface.co/datasets/vzc-research-chapter/VZCrash)
(Bianconcini et al., *VZCrash: A Large-Scale IMU Dataset of Ego-Vehicle Crashes*, IEEE ITSC 2026).
~190k events (>31k verified crashes), each a **16-second window** with:
- `gsensor` — tri-axial accelerometer, **g**, **100 Hz** (X fwd, Y left, Z up; ~1 g at rest)
- `gyro` — tri-axial, **deg/s**, 100 Hz (resolution depends on `gyro_is_hd`: 1 deg/s vs 0.001 deg/s)
- `gps_speed` — GPS-derived, **km/h**, **1 Hz**
- `label` — **0 = crash, 1 = near_miss, 2 = normal_driving** · `vehiclesize` · `gyro_is_hd`

`near_miss` (rapid evasive maneuver, no impact) is the **hard negative** — high-g/hard-braking events
that a naive threshold detector would false-positive on, so it's the key stressor for our rule.

> ⚖️ **License: CC BY-NC 4.0 (non-commercial).** Fine for research/validation like this. Do **not**
> redistribute the data, and treat thresholds/models *derived* from it as research findings — confirm
> with the maintainers/legal before baking VZCrash-derived parameters into a commercial release. Tracelet's
> own field collection should be the source for shipped defaults.

**Approach:** stream the events and reduce each to scalar features (peak g, Δv, gyro peak, …) — graphable,
not GBs of raw lists. The dataset repo also ships official `explore.ipynb` and `pr_curve.ipynb` (a physical
crash-detection baseline) worth comparing our sweep against.

What this notebook answers:
1. How separable are crashes from non-crashes (and from near-misses) on **peak g** alone?
2. How much does **Δv (speed drop)** and **gyro** add? (the #173 enhancements)
3. What false-negative / near-miss false-positive rate does Tracelet's current rule have?
4. What thresholds does the data actually recommend?
5. Does **downsampling 100 Hz → 5–10 Hz** miss the peak? (proves #172)

In [ ]:
!pip -q install datasets pandas numpy matplotlib scikit-learn

## 0. Authenticate (VZCrash is a *gated* dataset)

Two one-time prerequisites or you'll get `403 Forbidden`:
1. **Accept the terms** on the dataset page — open
   [huggingface.co/datasets/vzc-research-chapter/VZCrash](https://huggingface.co/datasets/vzc-research-chapter/VZCrash)
   while logged in and click **Agree and access repository**.
2. **Token permission:** your HF token must allow gated repos. In
   [Settings → Access Tokens](https://huggingface.co/settings/tokens), edit the token and enable
   **"Read access to contents of all public gated repos you can access"** (fine-grained), or use a
   classic **read** token. This is exactly what the `403 ... enable access to public gated repositories`
   error is telling you.

Then store it in Colab: **🔑 (left sidebar) → Secrets → add `HF_TOKEN`** with notebook access enabled.

In [ ]:
import os
from huggingface_hub import login

token = None
try:
    from google.colab import userdata          # Colab secret
    token = userdata.get('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN')          # local / other envs

assert token, 'No HF_TOKEN found — add it to Colab Secrets (key icon) and enable notebook access.'
login(token)
os.environ['HF_TOKEN'] = token                   # so `datasets` picks it up too
print('authenticated ok')

## 1. Peek at the schema (one streamed row)

In [ ]:
from datasets import load_dataset
import numpy as np, pandas as pd

REPO = "vzc-research-chapter/VZCrash"

# Streaming avoids downloading/decoding all 138k long-list rows up front.
stream = load_dataset(REPO, split="train", streaming=True)
first = next(iter(stream))

for k, v in first.items():
    if isinstance(v, list):
        arr = np.array(v, dtype=object)
        print(f"{k:12s} list len={len(v)} first={v[:2]}")
    else:
        print(f"{k:12s} {type(v).__name__} = {v}")

In [ ]:
# Enumerate the label classes (the viewer shows class 0 = 'crash').
info = load_dataset(REPO, split="train", streaming=True).info
try:
    print("label classes:", info.features["label"].names)
except Exception as e:
    print("could not read class names from features:", e)

## 2. Feature extraction

Reduce each event to scalars. **Units (confirmed from the dataset card):**
- `gsensor` — tri-axial accelerometer **in g** @ **100 Hz** (raw, so ‖a‖ ≈ 1 g at rest).
  We subtract the **1 g** gravity constant to match Tracelet's on-device `peak_g`.
- `gyro` — tri-axial **deg/s** @ 100 Hz (reported only; Tracelet doesn't threshold gyro yet — that's #173).
- `gps_speed` — GPS-derived speed **in km/h** @ **1 Hz** (matches Tracelet's `crash_min_speed_kmh`, also km/h).

The helpers stay robust to flat-vs-`[x,y,z]` shapes; the peek cell prints ranges so you can sanity-check
(`accel_hz_est` should read ~100, `speed_max` should look like real km/h).

In [ ]:
GRAVITY_G = 1.0     # VZCrash accelerometer is tri-axial in g, so 1 g of gravity is baked in.
WINDOW_S = 16.0     # every event is a fixed 16-second window (dataset card).

def magnitude_g(raw):
    """Gravity-subtracted acceleration magnitude (g), matching Tracelet's native
    `sqrt(x^2+y^2+z^2) - 1.0` (iOS) / `... - 9.81` then /9.81 (Android).

    VZCrash `gsensor` is raw tri-axial g (||a|| ~= 1 g at rest), so we subtract the
    1 g gravity constant — NOT a per-event mean — to replicate the device behaviour.
    Using the vector magnitude also makes it orientation-invariant, which matters
    because the card warns axis alignment is sometimes suboptimal.
    """
    arr = np.asarray(raw, dtype=float)
    if arr.ndim == 2 and arr.shape[-1] == 3:        # [x, y, z] samples (expected)
        mag = np.linalg.norm(arr, axis=1)
    else:                                            # already a magnitude series
        mag = np.abs(arr.ravel())
    if mag.size and np.median(mag) > 0.5:           # gravity present (~1 g) -> remove it
        mag = np.abs(mag - GRAVITY_G)
    return mag

def gyro_peak_dps(raw):
    """Peak gyroscope magnitude in deg/s (VZCrash gyro units)."""
    arr = np.asarray(raw, dtype=float)
    if arr.ndim == 2 and arr.shape[-1] == 3:
        return float(np.linalg.norm(arr, axis=1).max()) if len(arr) else 0.0
    return float(np.abs(arr).max()) if arr.size else 0.0

def features(ev):
    g = magnitude_g(ev["gsensor"])                  # g, gravity-subtracted
    spd = np.asarray(ev.get("gps_speed") or [], dtype=float)   # km/h @ 1 Hz
    n = len(g)
    return {
        "event_id": ev.get("event_id"),
        "label": ev.get("label"),
        "vehiclesize": ev.get("vehiclesize"),
        "gyro_is_hd": ev.get("gyro_is_hd"),         # gyro resolution: True=0.001, False=1 deg/s
        "peak_g": float(g.max()) if n else 0.0,     # g over baseline
        "mean_g": float(g.mean()) if n else 0.0,
        "gyro_peak_dps": gyro_peak_dps(ev.get("gyro") or []),  # deg/s
        "speed_max": float(spd.max()) if spd.size else 0.0,    # km/h
        "speed_min": float(spd.min()) if spd.size else 0.0,    # km/h
        "dv": float(spd.max() - spd.min()) if spd.size else 0.0,  # km/h speed drop
        "accel_n": n,
        "accel_hz_est": n / WINDOW_S,               # expect ~100 over the 16 s window
    }

In [ ]:
# Stream and reduce. Set MAX = None to process the full ~190k events (a few minutes);
# a sample is fine to start. Dataset: ~190k events, >31k verified crashes.
MAX = 20000

rows = []
for i, ev in enumerate(load_dataset(REPO, split="train", streaming=True)):
    try:
        rows.append(features(ev))
    except Exception:
        continue
    if MAX and i + 1 >= MAX:
        break

df = pd.DataFrame(rows)
# Map numeric label -> name if available.
try:
    names = info.features["label"].names
    df["label_name"] = df["label"].map(lambda i: names[i] if isinstance(i, int) and i < len(names) else str(i))
except Exception:
    df["label_name"] = df["label"].astype(str)
df["is_crash"] = df["label_name"].str.contains("crash", case=False, na=False)
print(f"events: {len(df)}")
print(df[["peak_g", "dv", "gyro_peak_dps", "speed_max", "accel_hz_est"]].describe())
print("\nclass balance:\n", df["label_name"].value_counts())

## 3. Graphs: how separable are crashes?

In [ ]:
import matplotlib.pyplot as plt

# Plot the 3 classes separately — near_miss is the hard negative (high g, no impact).
classes = [c for c in ['crash', 'near_miss', 'normal_driving'] if (df.label_name == c).any()]
colors = {'crash': 'tab:red', 'near_miss': 'tab:orange', 'normal_driving': 'tab:blue'}

fig, ax = plt.subplots(1, 3, figsize=(17, 4))

for c in classes:
    sub = df[df.label_name == c]
    ax[0].hist(sub.peak_g, bins=60, alpha=0.55, density=True, label=c, color=colors.get(c))
    ax[1].hist(sub.dv, bins=60, alpha=0.55, density=True, label=c, color=colors.get(c))
ax[0].axvline(3.0, color='k', ls='--', lw=1, label='Tracelet 3.0 g')
ax[0].set_title('peak g by class'); ax[0].set_xlabel('peak g (over baseline)'); ax[0].legend()
ax[1].set_title('dv (speed drop, km/h) by class'); ax[1].set_xlabel('dv'); ax[1].legend()

for c in classes:
    sub = df[df.label_name == c]
    ax[2].scatter(sub.peak_g, sub.dv, s=5, alpha=0.35, label=c, color=colors.get(c))
ax[2].axvline(3.0, color='k', ls='--', lw=1)
ax[2].set_title('peak g vs dv'); ax[2].set_xlabel('peak g'); ax[2].set_ylabel('dv (km/h)'); ax[2].legend()
plt.tight_layout(); plt.show()

## 4. Replay Tracelet's current rule and measure it

Tracelet (post-fix): a crash needs `peak_g >= crash_g_threshold` **and** pre-impact speed `>= crash_min_speed_kmh`.
Here `speed_max` proxies pre-impact speed.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

CRASH_G = 3.0          # ImpactConfig.crash_g_threshold
MIN_SPEED_KMH = 25.0   # ImpactConfig.crash_min_speed_kmh

pred = (df.peak_g >= CRASH_G) & (df.speed_max >= MIN_SPEED_KMH)
y = df.is_crash.values

tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
print(f"Tracelet rule (peak_g>={CRASH_G} g AND speed>={MIN_SPEED_KMH} km/h)")
print(f"  precision = {precision_score(y, pred):.3f}")
print(f"  recall    = {recall_score(y, pred):.3f}   <-- missed crashes = 1-recall")
print(f"  TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"  FALSE NEGATIVES (real crashes below threshold): {fn} / {tp+fn}")

# Where do the false positives come from? near_miss = expected hard negatives.
fp_mask = pred.values & ~y
fp_by_class = df.loc[fp_mask, "label_name"].value_counts()
print("\nfalse positives by true class (near_miss = hard negatives):")
print(fp_by_class)
for c in ["near_miss", "normal_driving"]:
    n_c = (df.label_name == c).sum()
    fpr = fp_by_class.get(c, 0) / n_c if n_c else float("nan")
    print(f"  {c:14s} FP rate = {fpr:.3f}  ({fp_by_class.get(c,0)}/{n_c})")

missed = df[y & ~pred.values]
print("\nwhy crashes were missed (sample):")
print(missed[["peak_g", "speed_max", "dv"]].describe())

## 5. Threshold sweep — what does the data recommend?

In [ ]:
ths = np.linspace(0.5, df.peak_g.quantile(0.999), 60)
prec, rec = [], []
for t in ths:
    p = (df.peak_g >= t) & (df.speed_max >= MIN_SPEED_KMH)
    prec.append(precision_score(y, p, zero_division=0))
    rec.append(recall_score(y, p, zero_division=0))

plt.figure(figsize=(7, 5))
plt.plot(ths, prec, label="precision")
plt.plot(ths, rec, label="recall")
plt.axvline(3.0, color="k", ls="--", lw=1, label="Tracelet 3.0 g")
plt.xlabel("crash_g_threshold"); plt.ylabel("score"); plt.legend()
plt.title("Precision/recall vs g-threshold (speed-gated)"); plt.grid(alpha=0.3); plt.show()

In [ ]:
# Does adding a Δv (speed-drop) corroboration help? (the #173 enhancement)
for dv_min in [0, 10, 20, 30]:
    p = (df.peak_g >= 3.0) & (df.speed_max >= MIN_SPEED_KMH) & (df.dv >= dv_min)
    print(f"dv>={dv_min:2d}:  precision={precision_score(y,p,zero_division=0):.3f}  recall={recall_score(y,p,zero_division=0):.3f}")

## 6. Sampling-rate check — does 5–10 Hz miss the peak? (proves #172)

Downsample a few crash events' high-rate `gsensor` to ~5 Hz and ~10 Hz and compare the captured peak
to the full-rate peak. If the low-rate peak frequently drops below threshold, that's the false-negative
mechanism #172 fixes.

In [ ]:
# Pull a handful of raw crash events again (streaming) to inspect the waveform + downsampling.
examples = []
for ev in load_dataset(REPO, split="train", streaming=True):
    g = magnitude_g(ev["gsensor"])
    if len(g) > 200 and g.max() > 2.0:
        examples.append((ev.get("event_id"), g))
    if len(examples) >= 3:
        break

FULL_HZ = 100  # confirmed by the dataset card (100 Hz accelerometer)

def downsample_peak(g, full_hz, target_hz):
    step = max(int(round(full_hz / target_hz)), 1)
    return g[::step].max()

fig, axes = plt.subplots(1, len(examples), figsize=(5*len(examples), 4))
if len(examples) == 1: axes = [axes]
for ax, (eid, g) in zip(axes, examples):
    ax.plot(g, lw=0.8)
    ax.axhline(3.0, color="k", ls="--", lw=1)
    p5 = downsample_peak(g, FULL_HZ, 5); p10 = downsample_peak(g, FULL_HZ, 10)
    ax.set_title(f"event {eid}\nfull peak={g.max():.1f}g  10Hz={p10:.1f}  5Hz={p5:.1f}")
    ax.set_xlabel("sample"); ax.set_ylabel("g over baseline")
plt.tight_layout(); plt.show()

## 7. (Optional) quick ML baseline — fusion beats a single threshold?

A small classifier on `[peak_g, dv, gyro_peak, speed_max, mean_g]` gives an upper bound on what sensor
fusion (#173) could achieve vs the single-threshold rule.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

feat_cols = ["peak_g", "dv", "gyro_peak_dps", "speed_max", "mean_g"]
X = df[feat_cols].fillna(0).values
clf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=0)
pred_cv = cross_val_predict(clf, X, y, cv=3)
print(classification_report(y, pred_cv, target_names=["non-crash", "crash"]))

clf.fit(X, y)
print("feature importance:", dict(zip(feat_cols, clf.feature_importances_.round(3))))

## Takeaways for #173

Fill in after running:
- Current rule recall (= 1 − missed-crash rate): **___**
- Recommended `crash_g_threshold` from the sweep: **___**
- Δv corroboration precision gain: **___**
- Low-rate (5–10 Hz) peak loss observed: **___** → confirms the high-rate fix (#172)
- Fusion (RF) recall vs single-threshold recall: **___** → sizing the ML opportunity

Feed the chosen thresholds back into `ImpactConfig` defaults and the website docs.